In [1]:
import pandas as pd
import numpy as np


# Content-Based: text similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler


# Collaborative Filtering: SVD matrix factorization
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy



# Evaluation metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
df = pd.read_excel("../Online_Course_recommendation_system/dataset/online_course.xlsx")
df.head()

,user_id,course_id,course_name,instructor,course_duration_hours,certification_offered,difficulty_level,rating,enrollment_numbers,course_price,feedback_score,study_material_available,time_spent_hours,previous_courses_taken
0,15796,9366,Python for Beginners,Emma Harris,39.1,Yes,Beginner,5.0,21600,317.50,0.797,Yes,17.60,4
1,861,1928,Cybersecurity for Professionals,Alexander Young,36.3,Yes,Beginner,4.3,15379,40.99,0.770,Yes,28.97,9
2,38159,9541,DevOps and Continuous Deployment,Dr. Mia Walker,13.4,Yes,Beginner,3.9,6431,380.81,0.772,Yes,52.44,4
3,44733,3708,Project Management Fundamentals,Benjamin Lewis,58.3,Yes,Beginner,3.1,48245,342.80,0.969,No,22.29,6
4,11285,3361,Ethical Hacking Masterclass,Daniel White,30.8,Yes,Beginner,2.8,34556,381.01,0.555,Yes,22.01,5


In [3]:
text_cols = ["course_name", "difficulty_level", "instructor",
             "certification_offered", "study_material_available"]
df[text_cols] = df[text_cols].fillna("")

In [4]:
df["rating"]  = df["rating"].fillna(df["rating"].median()).clip(1.0, 5.0)
print(f"Rows: {len(df)}")
print(f"Courses: {df['course_id'].nunique()}")
print(f"Users: {df['user_id'].nunique()}")


Rows: 100000
Courses: 9998
Users: 43242


## Content-Based: Here we Extraction Course Features 
*This measure how similar two courses are using TF-IDF + cosine similarity.*


In [5]:
# keep one row per course 
# dataset has many rows per course —> one per enrollment
course_cols = ["course_id","course_name","difficulty_level","instructor",
               "certification_offered","study_material_available",
               "course_duration_hours","course_price","feedback_score"]

courses = df[course_cols].drop_duplicates(subset="course_id").reset_index(drop=True)
courses[text_cols] = courses[text_cols].fillna("")
print(f"Unique courses: {len(courses)}")


Unique courses: 9998


In [6]:
# Scaling the numerical columns into numbers to [0,1] 
scaler   = MinMaxScaler()
num_cols = ["course_duration_hours", "course_price", "feedback_score"]
courses[num_cols] = courses[num_cols].fillna(df[num_cols].median())
courses[num_cols] = scaler.fit_transform(courses[num_cols])


In [7]:
# convert scaled numbers into text tokens so TF-IDF can similar them
for col, token in [("course_duration_hours","duration"), ("course_price","price"), ("feedback_score","feedback")]:
    courses[f"{token}_token"] = courses[col].apply(
        lambda x: f"{token}_high" if x > 0.66 else (f"{token}_mid" if x > 0.33 else f"{token}_low")
    )


In [8]:
# Combine everything into one feature string per course
# course_name repeated twice to give it more weight
courses["features"] = (
    courses["course_name"] + " " + courses["course_name"] + " " +
    courses["difficulty_level"] + " " + courses["instructor"] + " " +
    courses["certification_offered"] + " " + courses["study_material_available"] + " " +
    courses["duration_token"] + " " + courses["price_token"] + " " + courses["feedback_token"]
)

print("sample combination: ",courses["features"].iloc[0])

sample combination:  Python for Beginners Python for Beginners Beginner Emma Harris Yes Yes duration_mid price_mid feedback_high


In [9]:
# Build TF-IDF matrix — converts text to numbers the model can compare
tfidf        = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(courses["features"])
course_index = {cid: idx for idx, cid in enumerate(courses["course_id"])}

print(f"TF-IDF matrix = {tfidf_matrix.shape[0]} Rows -- courses = {tfidf_matrix.shape[1]} Features")


TF-IDF matrix = 9998 Rows -- courses = 411 Features


### Content-Based Score Calculation


In [10]:
#for a given user
#find courses they liked (rating ≥ 3.5)
#compute cosine similarity of those courses against all courses. 
#Average the similarities CB score per course.*


In [11]:
Sample_user = df["user_id"].iloc[0]
Min_rating  = 3.5

user_data = df[df["user_id"] == Sample_user]
liked_ids = user_data[user_data["rating"] >= Min_rating]["course_id"].tolist()

# fallback if no high ratings
if not liked_ids:
    liked_ids = user_data["course_id"].tolist()  

In [12]:
# Accumulate similarity scores across all liked courses
n = len(courses)
#creating the cb scores array
cb_raw = np.zeros(n)

for cid in liked_ids:
    if cid in course_index:
        idx     = course_index[cid]
        sim_row = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
        cb_raw += sim_row


In [13]:

if liked_ids:
    # average score
    cb_raw /= len(liked_ids)  

cb_scores = {cid: cb_raw[course_index[cid]] for cid in courses["course_id"] if cid in course_index}
print(f"CB scores ready for {len(cb_scores)} courses.")
print("Top 5 by CB score:", sorted(cb_scores.items(), key=lambda x: x[1], reverse=True)[:5])


CB scores ready for 9998 courses.
Top 5 by CB score: [(9366, 0.5516583211827479), (5928, 0.5516583211827479), (1741, 0.5516583211827479), (1849, 0.5442026016311345), (3368, 0.5342373551523696)]


## Collaborative filtering: Train SVD Model


In [14]:
cf_df = df[["user_id","course_id","rating"]].dropna()
cf_df["rating"] = cf_df["rating"].clip(1.0, 5.0)

reader   = Reader(rating_scale=(1.0, 5.0))
data_cf  = Dataset.load_from_df(cf_df[["user_id","course_id","rating"]], reader)
trainset = data_cf.build_full_trainset()

print(f"Trainset: {trainset.n_users} users --- {trainset.n_items} courses --- {trainset.n_ratings} ratings")


Trainset: 43242 users --- 9998 courses --- 100000 ratings


In [15]:
#Train SVD collaborative filtering model
cf_model = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
cf_model.fit(trainset)
print("SVD model trained.")


SVD model trained.


In [16]:
# Get CF predicted ratings for all courses for the sample user
all_course_ids = courses["course_id"].tolist()
cf_scores = {cid: cf_model.predict(Sample_user, cid).est for cid in all_course_ids}

print(f"CF scores ready for {len(cf_scores)} courses.")
print("Top 5 by CF score:", sorted(cf_scores.items(), key=lambda x: x[1], reverse=True)[:5])


CF scores ready for 9998 courses.
Top 5 by CF score: [(8086, 4.57618773702952), (8646, 4.570237453668733), (6014, 4.558036832571766), (2930, 4.54738131587234), (3861, 4.541719628939642)]


## Hybrid: Combine CB + CF Scores
*Before combining, we normalize both score types to [0,1] so scores on the same scale. hybrid_score = 0.4 × CB + 0.6 × CF




In [17]:
def normalize_scores(score_dict):
    """Min-Max normalize scores to [0,1] so CB and CF are on the same scale."""
    vals = np.array(list(score_dict.values()))
    mn, mx = vals.min(), vals.max()
    if mx == mn:
        return {k: 0.5 for k in score_dict}
    return {k: (v - mn) / (mx - mn) for k, v in score_dict.items()}


In [18]:
def hybrid_recommend(user_id, df, courses, cf_model, tfidf_matrix, course_index,
                     alpha=0.4, top_n=10, min_rating=3.5, switch_threshold=5):
    """
    Generate top-N hybrid recommendations.
    alpha = CB weight  |  (1-alpha) = CF weight
    if user has < 5 ratings, alpha is boosted to 0.7 .
    """
    seen = set(df[df["user_id"] == user_id]["course_id"])
    n_seen = len(seen)

    # new users get more CB weight
    eff_alpha = 0.7 if n_seen < switch_threshold else alpha
    print(f"User {user_id} | {n_seen} interactions | alpha={eff_alpha}")
#--------------------------------------------------------------------------------------------------------
    # CB scores
    user_liked = df[(df["user_id"]==user_id) & (df["rating"]>=min_rating)]["course_id"].tolist()
    if not user_liked:
        user_liked = df[df["user_id"]==user_id]["course_id"].tolist()
    
    # calculate Similarity
    cb_raw = np.zeros(len(courses))
    for cid in user_liked:
        if cid in course_index:
            cb_raw += cosine_similarity(tfidf_matrix[course_index[cid]], tfidf_matrix).flatten()
    
    if user_liked:
        #average score
        cb_raw /= len(user_liked)
    cb = {cid: cb_raw[course_index[cid]] for cid in courses["course_id"] if cid in course_index}
#------------------------------------------------------------------------------------------------------------------------
    
    # CF scores
    cf = {cid: cf_model.predict(user_id, cid).est for cid in courses["course_id"]}

    # Normalize & combine
    cb_n = normalize_scores(cb)
    cf_n = normalize_scores(cf)
    # alpha × CB_norm(c) + (1 - alpha) × CF_norm(c)
    hybrid = {cid: eff_alpha * cb_n.get(cid,0) + (1-eff_alpha) * cf_n.get(cid,0) for cid in courses["course_id"]}
#----------------------------------------------------------------------------------------------------
    # Remove seen, sort, top N
    unseen = {k: v for k, v in hybrid.items() if k not in seen}
    top    = sorted(unseen.items(), key=lambda x: x[1], reverse=True)[:top_n]

    rows = []
    for cid, hs in top:
        r = courses[courses["course_id"]==cid].iloc[0]
        rows.append({"course_id": cid, "course_name": r["course_name"],
                     "difficulty_level": r["difficulty_level"],
                     "cb_score": round(cb_n.get(cid,0),4),
                     "cf_score": round(cf_n.get(cid,0),4),
                     "hybrid_score": round(hs,4)})
    return pd.DataFrame(rows)


In [19]:
recs = hybrid_recommend(Sample_user, df, courses, cf_model, tfidf_matrix, course_index)
print(f"\nTop 10 Recommendations for User {Sample_user}")
recs


User 15796 | 2 interactions | alpha=0.7

Top 10 Recommendations for User 15796


,course_id,course_name,difficulty_level,cb_score,cf_score,hybrid_score
0,3368,Python for Beginners,Beginner,0.9684,0.6735,0.8799
1,5271,DevOps and Continuous Deployment,Beginner,0.9384,0.7241,0.8741
2,8867,DevOps and Continuous Deployment,Beginner,0.9027,0.7614,0.8603
3,2432,DevOps and Continuous Deployment,Beginner,0.9119,0.7074,0.8505
4,9348,DevOps and Continuous Deployment,Beginner,0.9067,0.7167,0.8497
5,1849,Python for Beginners,Beginner,0.9865,0.5222,0.8472
6,7531,Python for Beginners,Beginner,0.9508,0.6053,0.8472
7,5928,Python for Beginners,Beginner,1.0000,0.4499,0.8350
8,9185,DevOps and Continuous Deployment,Beginner,0.9210,0.6329,0.8346
9,6372,DevOps and Continuous Deployment,Beginner,0.8621,0.7558,0.8302


## Step 6 — Evaluation
*We evaluate using two approaches:*
ac- **Precision@K** — out of top K recommendations, how many were actually relevant? (higher = better)


In [20]:
#  RMSE & MAE:
# Split data into train/test (80/20)
train_data, test_data = train_test_split(data_cf, test_size=0.2, random_state=42)

# Train on 80 percent
svd_model = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
svd_model.fit(train_data)

# Predict on 20 percent
predictions = svd_model.test(test_data)

# Surprise built-in metrics
rmse = accuracy.rmse(predictions, verbose=False)
mae  = accuracy.mae(predictions,  verbose=False)

print(f"CF Model Evaluation (80/20 split)")
print(f"  RMSE : {rmse:.4f}  (avg error in rating prediction, scale 1-5)")
print(f"  MAE  : {mae:.4f}  (mean absolute error)")


CF Model Evaluation (80/20 split)
  RMSE : 0.7445  (avg error in rating prediction, scale 1-5)
  MAE  : 0.6105  (mean absolute error)


In [21]:
valid_users = df["user_id"].unique()
user_id_input = int(input(f"Enter User ID (valid range {valid_users.min()}–{valid_users.max()}): "))

if user_id_input not in valid_users:
    print(f"User {user_id_input} not found in dataset.")
else:
    result = hybrid_recommend(user_id_input, df, courses, cf_model,
                              tfidf_matrix, course_index, alpha=0.4, top_n=10)
    print(f"\nTop 10 Recommendations for User {user_id_input}")
    display(result)


Enter User ID (valid range 1–49999):  4345


User 4345 | 4 interactions | alpha=0.7

Top 10 Recommendations for User 4345


,course_id,course_name,difficulty_level,cb_score,cf_score,hybrid_score
0,7947,Stock Market and Trading Strategies,Beginner,1.0000,0.7423,0.9227
1,3354,Fitness and Nutrition Coaching,Advanced,0.9434,0.8079,0.9028
2,1733,Blockchain and Decentralized Applications,Beginner,0.9315,0.8350,0.9026
3,5473,Fitness and Nutrition Coaching,Beginner,0.9257,0.8365,0.8990
4,8569,Fitness and Nutrition Coaching,Advanced,0.9434,0.7686,0.8910
5,1770,Fitness and Nutrition Coaching,Beginner,0.9230,0.7863,0.8820
6,4918,Fitness and Nutrition Coaching,Beginner,0.9315,0.7444,0.8754
7,9877,Stock Market and Trading Strategies,Beginner,0.9263,0.7499,0.8734
8,7341,Blockchain and Decentralized Applications,Beginner,0.9047,0.7858,0.8691
9,5863,Fitness and Nutrition Coaching,Beginner,0.9411,0.6995,0.8687


In [22]:
import pickle

hybrid_model = {
    "cf_model"      : cf_model,
    "tfidf_matrix"  : tfidf_matrix,
    "course_index"  : course_index,
    "courses"       : courses,
    "alpha"         : 0.4,
    "switch_threshold": 5,
    "min_rating"    : 3.5
}

with open("hybrid_model.pkl", "wb") as f:
    pickle.dump(hybrid_model, f)

print("Model saved to hybrid_model.pkl")

Model saved to hybrid_model.pkl


In [23]:
with open("hybrid_model.pkl", "rb") as f:
    hybrid_model = pickle.load(f)

# Unpack components
cf_model       = hybrid_model["cf_model"]
tfidf_matrix   = hybrid_model["tfidf_matrix"]
course_index   = hybrid_model["course_index"]
courses        = hybrid_model["courses"]
alpha          = hybrid_model["alpha"]

In [24]:
recs = hybrid_recommend(
    user_id         = 42,
    df              = df,
    courses         = hybrid_model["courses"],
    cf_model        = hybrid_model["cf_model"],
    tfidf_matrix    = hybrid_model["tfidf_matrix"],
    course_index    = hybrid_model["course_index"],
    alpha           = hybrid_model["alpha"],
    top_n           = 10
)

User 42 | 1 interactions | alpha=0.7


In [25]:
display(result)

,course_id,course_name,difficulty_level,cb_score,cf_score,hybrid_score
0,7947,Stock Market and Trading Strategies,Beginner,1.0000,0.7423,0.9227
1,3354,Fitness and Nutrition Coaching,Advanced,0.9434,0.8079,0.9028
2,1733,Blockchain and Decentralized Applications,Beginner,0.9315,0.8350,0.9026
3,5473,Fitness and Nutrition Coaching,Beginner,0.9257,0.8365,0.8990
4,8569,Fitness and Nutrition Coaching,Advanced,0.9434,0.7686,0.8910
5,1770,Fitness and Nutrition Coaching,Beginner,0.9230,0.7863,0.8820
6,4918,Fitness and Nutrition Coaching,Beginner,0.9315,0.7444,0.8754
7,9877,Stock Market and Trading Strategies,Beginner,0.9263,0.7499,0.8734
8,7341,Blockchain and Decentralized Applications,Beginner,0.9047,0.7858,0.8691
9,5863,Fitness and Nutrition Coaching,Beginner,0.9411,0.6995,0.8687
